In [10]:
import time
import pandas as pd
import numpy as np
from pmdarima import auto_arima
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")
data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)
data = data.set_index("time")
data.index.freq = "h"

split = int(np.ceil(0.8 * len(data)))
train = data.iloc[:split]
test  = data.iloc[split:]

# --- Train ---
start = time.time()
model = auto_arima(
    train["pm2_5"],
    seasonal=True, m=24, stepwise=True,
    information_criterion="aic",
    max_p=2, max_q=2,
    max_P=1, max_D=1, max_Q=1,
)
training_time = time.time() - start

In [ ]:
test_pm25 = test["pm2_5"].reset_index(drop=True)

preds = []
for i in range(len(test_pm25)):
    preds.append(model.predict(n_periods=1).item())
    model.update(test_pm25.iloc[[i]])

# --- Inference time on single instance ---
single_start = time.time()
model.predict(n_periods=1).item()
inference_time = time.time() - single_start

# --- Inference time on single instance (predict + update) ---
single_start = time.time()
model.predict(n_periods=1).item()
model.update(test_pm25.iloc[[0]])
update_inference_time = time.time() - single_start

preds = np.array(preds)

rmse = root_mean_squared_error(test_pm25, preds)
mae  = mean_absolute_error(test_pm25, preds)
r2   = r2_score(test_pm25, preds)

print(f"RMSE          : {rmse:.4f}")
print(f"MAE           : {mae:.4f}")
print(f"R2            : {r2:.4f}")
print(f"Training time : {training_time:.2f}s")
print(f"Inference time (predict only)  :b
 {inference_time*1_000_000:.4f}μs")
print(f"Inference time (predict+update): {update_inference_time*1_000_000:.4f}μs")

RMSE          : 4.7298
MAE           : 2.4223
R2            : 0.9750
Training time : 1024.34s
Inference time (predict only)  : 7518.7683μs
Inference time (predict+update): 9316920.5189μs


In [12]:
import joblib

joblib.dump(model, "sarima_model.pkl")

['sarima_model.pkl']